# Raviart–Thomas Darcy: RT0, RT1 and RT2

The MHM flux is locally RTm and its pressure is discontinuous Pm. The classical comparison is globally conforming RTm/Pm on the same fine spacing. Physical flux, pressure and divergence norms below use the exact sinusoidal solution.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
from IPython.display import Image, display
from pymhm.mesh import TriangleMesh
from pymhm.darcy_rt import solve_darcy_rt
root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent


## Anisotropic affine patch

This check exercises physical flux orientation and exact RT representation; it is separate from the convergence study.


In [ ]:
K = np.array([[2.0, 0.3], [0.3, 1.0]])
def pressure(x):
    """Return an affine pressure with nonhomogeneous boundary data."""
    return 1 + x[:, 0] + 2 * x[:, 1]
for degree in (0, 1, 2):
    solution = solve_darcy_rt(TriangleMesh.unit_square(1), degree=degree, permeability=K, dirichlet=pressure)
    assert solution.flux_l2_error(-K @ np.array([1.0, 2.0])) < 1e-10
    assert max(np.max(np.abs(r)) for r in solution.fine_equilibrium_residuals()) < 1e-10
print("All three RT flux patches passed")


## Five levels against the analytical solution

The archived campaign has local refinement two and one degree-m polynomial per macroface. The classical method has no macroface restriction. Its own error is measured against the exact field; it is not treated as exact.


In [ ]:
record = json.loads((root / "examples/results/darcy-rt.json").read_text())
assert not record["source_changed_during_run"]
assert len(record["rows"]) == 15
for degree in (0, 1, 2):
    rows = [row for row in record["rows"] if row["degree"] == degree]
    assert [row["n"] for row in rows] == [1, 2, 4, 8, 16]
    for method in ("mhm", "classical"):
        assert rows[-1][method]["flux_l2_error"] < rows[0][method]["flux_l2_error"]
        assert max(row[method]["fine_moment_residual"] for row in rows) < 1e-10
    print(degree, rows[-1]["mhm"], rows[-1]["classical"])
last = record["rows"][-1]
archive = root / "examples/results" / last["fields"]
assert hashlib.sha256(archive.read_bytes()).hexdigest() == last["fields_sha256"]


In [ ]:
display(Image(filename=str(root / "docs/figures/darcy-rt/convergence.png")))
display(Image(filename=str(root / "docs/figures/darcy-rt/fields.png")))


The field panels show individual fine-cell centroid samples and actual macro edges. Volume norms use complete RT/Pm polynomials, preserving discontinuous pressure and one-sided material values. Native optional tests independently assemble RT0–RT2 with DOLFINx/UFL.
